In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


In [6]:
X, y, true_coefs = make_regression(
    n_samples   = 150,      
    n_features  = 30,       
    n_informative = 10,     
    noise       = 25,      
    coef        = True,     
    random_state= 42
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [22]:
feat_names = [f"F{i+1:02d}" for i in range(X.shape[1])]


In [8]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test) 

In [9]:
alphas = np.logspace(-3, 3, 60)

In [17]:
ridge_coefs = []   # har alpha ke liye Ridge ke 30 weights
lasso_coefs = []   # har alpha ke liye Lasso ke 30 weights

ridge_metrics = {"MAE": [], "RMSE": [], "R2": []}
lasso_metrics = {"MAE": [], "RMSE": [], "R2": []}

In [18]:
for a in alphas:
    # Ridge
    r = Ridge(alpha=a)
    r.fit(X_train_s, y_train)
    ridge_coefs.append(r.coef_)
 
    # Lasso
    l = Lasso(alpha=a, max_iter=10000)
    l.fit(X_train_s, y_train)
    lasso_coefs.append(l.coef_)
 
ridge_coefs = np.array(ridge_coefs)   
lasso_coefs = np.array(lasso_coefs)   
 
print(f"Ridge paths computed : {ridge_coefs.shape}")
print(f"Lasso paths computed : {lasso_coefs.shape}")

lr = LinearRegression()
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)

Ridge paths computed : (60, 30)
Lasso paths computed : (60, 30)


In [19]:
lr_metrics = {
    "MAE" : mean_absolute_error(y_test, lr_pred),
    "RMSE": root_mean_squared_error(y_test, lr_pred),
    "R2"  : r2_score(y_test, lr_pred)
}

print(f"\nLinear Regression → MAE: {lr_metrics['MAE']:.2f} | "
      f"RMSE: {lr_metrics['RMSE']:.2f} | R²: {lr_metrics['R2']:.4f}")


Linear Regression → MAE: 26.25 | RMSE: 32.74 | R²: 0.9527


In [20]:
for i, a in enumerate(alphas):
    # Ridge predict karo stored coefficients se
    r_pred = X_test_s @ ridge_coefs[i] + Ridge(alpha=a).fit(X_train_s, y_train).intercept_
    ridge_metrics["MAE"].append(mean_absolute_error(y_test, r_pred))
    ridge_metrics["RMSE"].append(root_mean_squared_error(y_test, r_pred))
    ridge_metrics["R2"].append(r2_score(y_test, r_pred))
 
    # Lasso predict karo
    l_pred = X_test_s @ lasso_coefs[i] + Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train).intercept_
    lasso_metrics["MAE"].append(mean_absolute_error(y_test, l_pred))
    lasso_metrics["RMSE"].append(root_mean_squared_error(y_test, l_pred))
    lasso_metrics["R2"].append(r2_score(y_test, l_pred))

In [25]:
lasso_cv = LassoCV(
    alphas  = alphas,
    cv      = 5,
    max_iter= 10000
)
lasso_cv.fit(X_train_s, y_train)
 
best_alpha    = lasso_cv.alpha_
best_coefs    = lasso_cv.coef_
zero_features = [feat_names[i] for i, c in enumerate(best_coefs) if c == 0]
active_feats  = [feat_names[i] for i, c in enumerate(best_coefs) if c != 0]

lasso_cv_pred = lasso_cv.predict(X_test_s)
lasso_cv_metrics = {
    "MAE" : mean_absolute_error(y_test, lasso_cv_pred),
    "RMSE": root_mean_squared_error(y_test, lasso_cv_pred),
    "R2"  : r2_score(y_test, lasso_cv_pred)
}

In [26]:
print(f"\nBest alpha (LassoCV) : {best_alpha:.5f}")
print(f"Features eliminated  : {len(zero_features)} out of 30")
print(f"Features kept        : {len(active_feats)}")
print(f"Eliminated features  : {', '.join(zero_features)}")
print(f"Active features      : {', '.join(active_feats)}")
print(f"\nLassoCV Metrics → MAE: {lasso_cv_metrics['MAE']:.2f} | "
      f"RMSE: {lasso_cv_metrics['RMSE']:.2f} | R²: {lasso_cv_metrics['R2']:.4f}")


Best alpha (LassoCV) : 1.42083
Features eliminated  : 11 out of 30
Features kept        : 19
Eliminated features  : F02, F05, F08, F09, F11, F12, F13, F21, F22, F25, F26
Active features      : F01, F03, F04, F06, F07, F10, F14, F15, F16, F17, F18, F19, F20, F23, F24, F27, F28, F29, F30

LassoCV Metrics → MAE: 23.15 | RMSE: 29.10 | R²: 0.9626


In [28]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plt.suptitle("Regularization Analysis: Ridge vs Lasso", fontsize=14, fontweight="bold")
 
# ── Plot 1: Ridge coefficient path ──
ax1 = axes[0, 0]
ax1.plot(alphas, ridge_coefs, color="steelblue", alpha=0.5, lw=1)
ax1.axhline(0, color="black", lw=0.8)
ax1.set_xscale("log")
ax1.set_title("Ridge — Coefficient Path")
ax1.set_xlabel("Alpha")
ax1.set_ylabel("Coefficient value")
ax1.grid(True, alpha=0.3)
 
# ── Plot 2: Lasso coefficient path ──
ax2 = axes[0, 1]
ax2.plot(alphas, lasso_coefs, alpha=0.5, lw=1)
ax2.axhline(0, color="black", lw=0.8)
ax2.axvline(best_alpha, color="red", lw=1.5, linestyle="--", label=f"Best α={best_alpha:.3f}")
ax2.set_xscale("log")
ax2.set_title("Lasso — Coefficient Path")
ax2.set_xlabel("Alpha")
ax2.set_ylabel("Coefficient value")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
 
# ── Plot 3: R² vs Alpha ──
ax3 = axes[1, 0]
ax3.axhline(lr_metrics["R2"], color="orange", lw=1.5, linestyle=":", label=f"Linear Reg R²={lr_metrics['R2']:.3f}")
ax3.plot(alphas, ridge_metrics["R2"], color="steelblue", lw=2, label="Ridge")
ax3.plot(alphas, lasso_metrics["R2"], color="green",     lw=2, label="Lasso")
ax3.axvline(best_alpha, color="red", lw=1.5, linestyle="--", label="LassoCV best α")
ax3.set_xscale("log")
ax3.set_title("R² vs Alpha")
ax3.set_xlabel("Alpha")
ax3.set_ylabel("R²")
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
 
# ── Plot 4: RMSE vs Alpha ──
ax4 = axes[1, 1]
ax4.axhline(lr_metrics["RMSE"], color="orange", lw=1.5, linestyle=":", label=f"Linear Reg RMSE={lr_metrics['RMSE']:.1f}")
ax4.plot(alphas, ridge_metrics["RMSE"], color="steelblue", lw=2, label="Ridge")
ax4.plot(alphas, lasso_metrics["RMSE"], color="green",     lw=2, label="Lasso")
ax4.axvline(best_alpha, color="red", lw=1.5, linestyle="--", label="LassoCV best α")
ax4.set_xscale("log")
ax4.set_title("RMSE vs Alpha")
ax4.set_xlabel("Alpha")
ax4.set_ylabel("RMSE")
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig("regularization_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot saved → regularization_analysis.png")
 

Plot saved → regularization_analysis.png


In [29]:
best_ridge_idx = np.argmin(ridge_metrics["RMSE"])
best_ridge_alpha = alphas[best_ridge_idx]
 
# Best Lasso (lowest RMSE wala alpha)
best_lasso_idx = np.argmin(lasso_metrics["RMSE"])
best_lasso_alpha = alphas[best_lasso_idx]
 
summary_data = {
    "Model"           : ["Linear Regression", f"Ridge (α={best_ridge_alpha:.3f})", f"Lasso (α={best_lasso_alpha:.3f})", f"LassoCV (α={best_alpha:.4f})"],
    "MAE"             : [
        lr_metrics["MAE"],
        ridge_metrics["MAE"][best_ridge_idx],
        lasso_metrics["MAE"][best_lasso_idx],
        lasso_cv_metrics["MAE"]
    ],
    "RMSE"            : [
        lr_metrics["RMSE"],
        ridge_metrics["RMSE"][best_ridge_idx],
        lasso_metrics["RMSE"][best_lasso_idx],
        lasso_cv_metrics["RMSE"]
    ],
    "R²"              : [
        lr_metrics["R2"],
        ridge_metrics["R2"][best_ridge_idx],
        lasso_metrics["R2"][best_lasso_idx],
        lasso_cv_metrics["R2"]
    ],
    "Zero Coefs"      : [0, 0, int(np.sum(lasso_coefs[best_lasso_idx] == 0)), len(zero_features)],
    "Active Features" : [30, 30, int(np.sum(lasso_coefs[best_lasso_idx] != 0)), len(active_feats)],
}

In [30]:
df_summary = pd.DataFrame(summary_data)
df_summary["MAE"]  = df_summary["MAE"].round(2)
df_summary["RMSE"] = df_summary["RMSE"].round(2)
df_summary["R²"]   = df_summary["R²"].round(4)
 
print("\n")
print(df_summary.to_string(index=False))
print("\n")
 
# Best model highlight
best_model_idx = df_summary["R²"].idxmax()
print(f"★  Best model by R²  : {df_summary.loc[best_model_idx, 'Model']}")
print(f"★  Best R²           : {df_summary.loc[best_model_idx, 'R²']}")
print(f"★  Best RMSE         : {df_summary.loc[best_model_idx, 'RMSE']}")
print(f"\n★  LassoCV eliminated {len(zero_features)} noise features automatically")
print(f"★  True informative features in dataset : 10")
print(f"★  LassoCV kept : {len(active_feats)} features")
print("\n" + "=" * 60)



             Model   MAE  RMSE     R²  Zero Coefs  Active Features
 Linear Regression 26.25 32.74 0.9527           0               30
   Ridge (α=0.001) 26.25 32.74 0.9527           0               30
   Lasso (α=2.270) 23.50 28.81 0.9633          15               15
LassoCV (α=1.4208) 23.15 29.10 0.9626          11               19


★  Best model by R²  : Lasso (α=2.270)
★  Best R²           : 0.9633
★  Best RMSE         : 28.81

★  LassoCV eliminated 11 noise features automatically
★  True informative features in dataset : 10
★  LassoCV kept : 19 features

